<a href="https://colab.research.google.com/github/aadhyabansal/nano_gpt/blob/main/nano_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#downloading dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-08 09:39:51--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-07-08 09:39:52 (22.5 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
#reading dataset
with open("input.txt",'r',encoding='utf-8') as f:
    text=f.read()
print(len(text))
text[:100]

1115394


'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [3]:
#creating vocabulary
chars=sorted(list(set(text)))
vocab_size=len(chars)
print(''.join(chars))
vocab_size


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


65

In [4]:
#encoding-decoding text
stoi= {ch:i for i, ch in enumerate(chars)}
itos= {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: ''.join(itos[i] for i in l)

print(encode("hello world"))
decode(encode("hello world"))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]


'hello world'

In [5]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

In [6]:
#converting data in tensors
data=torch.tensor(encode(text))

In [7]:
#splitting data
n=int(0.9*len(data))
train_data=data[:n]
val_data=data[n:]

In [18]:
#hyperparameters

batch_size=64
block_size=256
max_iters=5000
eval_interval=500
eval_iters=200
lr=3e-4
n_embd=384
n_head=6
n_layer=6
dropout=0.2
device='cuda' if torch.cuda.is_available() else 'cpu'


In [19]:
#creating a batch
torch.manual_seed(1337)

def get_batch(split):
    data= train_data if split=="train" else val_data
    ix=torch.randint(len(data)-block_size,(batch_size,))
    x=torch.stack([data[i:i+block_size] for i in ix])
    y=torch.stack([data[i+1:i+block_size+1] for i in ix])
    x,y=x.to(device),y.to(device)
    return x,y


In [20]:
#estimating loss

@torch.no_grad()
def estimate_loss():
    out={}
    model.eval()
    for split in ['train','val']:
        losses=torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y=get_batch(split)
            logits,loss=model(X,Y)
            losses[k]=loss.item()
        out[split]=losses.mean()
    model.train()
    return out

In [28]:
#attention block
class Head(nn.Module):

    def __init__(self,head_size):
        super().__init__()
        self.key=nn.Linear(n_embd,head_size,bias=False)
        self.query=nn.Linear(n_embd,head_size,bias=False)
        self.value=nn.Linear(n_embd,head_size,bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))
        self.head_size= head_size
        self.dropout=nn.Dropout(dropout)

    def forward(self,x):
        B,T,C=x.shape
        k=self.key(x)
        q=self.query(x)

        wei= q @ k.transpose(-2,-1)
        wei= wei.masked_fill(self.tril[:T,:T]==0, float('-inf'))
        wei=F.softmax(wei, dim=-1) * self.head_size** -0.5
        wei= self.dropout(wei)

        v=self.value(x)

        out=wei @ v
        return out

In [29]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads=nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj=nn.Linear(n_embd,n_embd)

        self.dropout=nn.Dropout(dropout)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [30]:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self,x):
        return self.net(x)


In [31]:
class Block(nn.Module):

    def __init__(self, n_embd, num_head):
        super().__init__()
        head_size = n_embd // num_head
        self.sa_head=MultiHeadAttention(4,n_embd//4)
        self.ffwd=FeedForward(n_embd)
        self.ln1=nn.LayerNorm(n_embd)
        self.ln2=nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa_head(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


In [32]:
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,n_embd)
        self.positional_embedding_table=nn.Embedding(block_size,n_embd)
        self.blocks= nn.Sequential(*[Block(n_embd, num_head=n_head) for _ in range(n_layer)])
        self.ln_f=nn.LayerNorm(n_embd)
        self.lm_head=nn.Linear(n_embd,vocab_size)

    def forward(self,idx,targets=None):

        B,T=idx.shape
        tokn_embd=self.token_embedding_table(idx)
        pos_embd=self.positional_embedding_table(torch.arange(T, device=device))
        x = tokn_embd + pos_embd
        x = self.blocks(x)
        x = self.ln_f(x)
        logits=self.lm_head(x)

        if targets==None:
            loss=None
        else:
            B,T,C=logits.shape
            logits=logits.view(B*T,C)
            targets=targets.view(B*T)
            loss= F.cross_entropy(logits,targets)
        return logits, loss

    def generate(self,idx,max_new_tokens):
        for i in range(max_new_tokens):
            idx_cond=idx[:,-block_size:]
            logits,loss=self(idx_cond)
            logits=logits[:,-1,:]
            probs=F.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
            idx=torch.cat((idx,idx_next),dim=1)
        return idx


model=BigramLanguageModel()
model = model.to(device)
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

10.788929 M parameters


In [33]:
#creating an optimiser
optimizer=torch.optim.AdamW(model.parameters(),lr=lr)

In [34]:
#calculating loss
for i in range(max_iters):
    if i % eval_interval==0 or i==max_iters-1 :
        losses=estimate_loss()
        print(f"step {i} : train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb,yb=get_batch('train')
    logits,loss=model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0 : train loss 4.3183, val loss 4.3169
step 500 : train loss 2.0306, val loss 2.0948
step 1000 : train loss 1.6898, val loss 1.8320
step 1500 : train loss 1.5318, val loss 1.7111
step 2000 : train loss 1.4353, val loss 1.6329
step 2500 : train loss 1.3707, val loss 1.5870
step 3000 : train loss 1.3214, val loss 1.5515
step 3500 : train loss 1.2863, val loss 1.5286
step 4000 : train loss 1.2561, val loss 1.5109
step 4500 : train loss 1.2312, val loss 1.4990
step 4999 : train loss 1.2059, val loss 1.4973


In [35]:
context= torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=1000)[0].tolist()))



KING RICHARD III:
Where you thine is as sydes, Lord Aumerle;
Are two of Pompeixt, bid within quarrel,
Who knows by thine, I do be smiles.

LADY GREY:
God go, our brooks the strong Bushy, quarench
To our husband the Duke of York you are.

DUKE VINCENTIO:
Since on the north, of our prayer yet Master father,
Driving with out.

HENRY CLIFFORCAS:
You have rech done and chall setty ingrow man
The time for you.
Nays! what with your scener, this my lord,
I mean in all in thee with all voice.

Lady:
You noble hate conto you? he ha shall please
The arms, 'tis hour anguise true: a justice can happy
tooth, when I'll pluck'd that hath,--I have the depened
A man a Jack, it with the two halfs! us
This is true-hood. Why, you'll save Richmond it.

OXFORXight:
Why hast Katham! Verona call,
When my child, he deniolemen:
And no wheneyed when the mind: and the power,
By trumpets to my browl'd, I'll bear this now it.

KING RICHmin:
Sir, what you within wedd I have jove they
The bitter, which you would giv